In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load dataset
data = pd.read_csv('/content/drive/MyDrive/DAYTON_hourly.csv')
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.set_index('Datetime', inplace=True)

data.sort_values(by='Datetime', inplace=True)

# Normalize data (hourly data)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data['DAYTON_MW'].values.reshape(-1, 1))

# Create sequences for supervised learning
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return np.array(X), np.array(y)

lookback = 30  # Use the last 30 hours to predict the next hour
X, y = create_sequences(scaled_data, lookback)

# Create train, validation, and test splits
timestamps = len(X)
train_size = int(0.7 * timestamps)  # 70% for training
val_size = int(0.1 * timestamps)    # 10% for validation
test_size = timestamps - train_size - val_size  # Remaining for testing

X_train, X_val, X_test = X[:train_size], X[train_size:train_size+val_size], X[train_size+val_size:]
y_train, y_val, y_test = y[:train_size], y[train_size:train_size+val_size], y[train_size+val_size:]

# Print shapes for verification
print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")


X_train shape: (84871, 30, 1), X_val shape: (12124, 30, 1), X_test shape: (24250, 30, 1)


## GRU

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Define the GRU model
gru_model = Sequential([
    GRU(50, activation='relu', input_shape=(lookback, 1)),
    Dense(1)  # Single output
])

# Compile the model
gru_model.compile(optimizer='adam', loss = 'mae', metrics = ['mse', 'mae'])
print(gru_model.summary())

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                          │ (None, 50)                  │           7,950 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,001 (31.25 KB)

 Trainable params: 8,001 (31.25 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the GRU model
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,  # High number to allow early stopping
    batch_size=32,
    callbacks=[early_stopping]
)


Epoch 1/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0617 - mae: 0.0617 - mse: 0.0111 - val_loss: 0.0147 - val_mae: 0.0147 - val_mse: 3.8507e-04
Epoch 2/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0146 - mae: 0.0146 - mse: 3.8186e-04 - val_loss: 0.0130 - val_mae: 0.0130 - val_mse: 2.8652e-04
Epoch 3/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.0123 - mae: 0.0123 - mse: 2.7784e-04 - val_loss: 0.0108 - val_mae: 0.0108 - val_mse: 2.1278e-04
Epoch 4/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0115 - mae: 0.0115 - mse: 2.5272e-04 - val_loss: 0.0110 - val_mae: 0.0110 - val_mse: 2.1772e-04
Epoch 5/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0114 - mae: 0.0114 - mse: 2.4880e-04 - val_loss: 0.0108 - val_mae: 0.0108 - val_mse: 2.1524e-04
Epoch 6/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0112 - mae: 0.0112 - mse: 2.4261e-04 - val_loss: 0.0106 - val_mae: 0.0106 - val_mse: 2.0785e-04
Epoch 7/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
# Evaluate the model on the test set
gru_loss = gru_model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {gru_loss}")

above should be MAE, not MSE

In [ ]:
# Predict on the test set
gru_predictions = gru_model.predict(X_test)

# Invert scaling for predictions and true values
gru_predictions_original = scaler.inverse_transform(gru_predictions)
y_test_original = scaler.inverse_transform(y_test)

# Evaluate performance in the original scale
from sklearn.metrics import mean_absolute_error, mean_squared_error

gru_mae = mean_absolute_error(y_test_original, gru_predictions_original)
gru_rmse = np.sqrt(mean_squared_error(y_test_original, gru_predictions_original))

print(f"GRU - Mean Absolute Error (MAE): {gru_mae:.4f} MW")
print(f"GRU - Root Mean Squared Error (RMSE): {gru_rmse:.4f} MW")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test_original, label='Actual', alpha=0.8)
plt.plot(gru_predictions_original, label='Predicted', linestyle='--', alpha=0.8)
plt.title('GRU Predictions vs Actual Values (Original Scale)')
plt.xlabel('Time Step')
plt.ylabel('Electricity Consumption (MW)')
plt.legend()
plt.grid()
plt.show()


## Transformers

In [ ]:
# ----------------------------------------
# 1. Install/Import Dependencies
# ----------------------------------------
# !pip install tensorflow numpy pandas scikit-learn matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, MultiHeadAttention, LayerNormalization,
    Dropout, Flatten
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ----------------------------------------
# 2. Load and Sort the Data
# ----------------------------------------
# Make sure you've downloaded 'PJMW_hourly.csv' from Kaggle / PJM dataset
file_path = '/content/drive/MyDrive/PJMW_hourly.csv'  # Adjust to your path

df = pd.read_csv(file_path)
df['Datetime'] = pd.to_datetime(df['Datetime'])
df.sort_values(by='Datetime', inplace=True)  # Ensure chronological order
df.set_index('Datetime', inplace=True)

# ----------------------------------------
# 3. Scale the Data
# ----------------------------------------
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(df['PJMW_MW'].values.reshape(-1, 1))

# ----------------------------------------
# 4. Create Sequences for Supervised Learning
#    (lookback hours -> next hour)
# ----------------------------------------
def create_sequences(data, lookback=30):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])  # sequence of length 'lookback'
        y.append(data[i+lookback])    # the next time-step value
    return np.array(X), np.array(y)

lookback = 30
X_all, y_all = create_sequences(scaled_values, lookback=lookback)

print("X_all shape:", X_all.shape)  # (N, 30, 1)
print("y_all shape:", y_all.shape)  # (N,)

# ----------------------------------------
# 5. Split into Train / Validation / Test
# ----------------------------------------
timestamps = len(X_all)
train_size = int(0.7 * timestamps)  # 70% train
val_size   = int(0.1 * timestamps)  # 10% validation
test_size  = timestamps - train_size - val_size  # 20% test

X_train = X_all[:train_size]
y_train = y_all[:train_size]

X_val = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size]

X_test = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:]

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape}, {y_val.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")

# ----------------------------------------
# 6. Define a Transformer Encoder Block
# ----------------------------------------
def transformer_encoder(
    inputs,
    num_heads=4,
    key_dim=8,
    dff=128,
    dropout_rate=0.1
):
    """
    A single Transformer encoder block:
    - Multi-head self-attention
    - Add & Layer Norm
    - Feedforward
    - Add & Layer Norm
    """
    # Multi-head self-attention
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout_rate
    )(inputs, inputs)  # self-attn: queries=keys=values=inputs

    # Residual connection + Dropout + Layer Norm
    x = Dropout(dropout_rate)(attention_output)
    x = LayerNormalization(epsilon=1e-6)(inputs + x)

    # Feedforward
    ffn = Dense(dff, activation='relu')(x)
    ffn = Dense(x.shape[-1])(ffn)  # project back to model dimension

    # Residual connection + Dropout + Layer Norm
    ffn_out = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn_out)

# ----------------------------------------
# 7. Build the Transformer Model
# ----------------------------------------
def build_transformer_model(
    lookback=30,
    num_heads=4,
    key_dim=8,
    dff=128,
    dropout_rate=0.1
):
    """
    Minimal Transformer model for time-series forecasting.
    Input shape: (batch, lookback, 1)
    """
    inputs = Input(shape=(lookback, 1))
    x = transformer_encoder(
        inputs,
        num_heads=num_heads,
        key_dim=key_dim,
        dff=dff,
        dropout_rate=dropout_rate
    )
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs, name='TransformerForecast')
    return model

# ----------------------------------------
# 8. Instantiate and Compile
# ----------------------------------------
model = build_transformer_model(
    lookback=lookback,
    num_heads=4,
    key_dim=8,
    dff=128,
    dropout_rate=0.1
)

model.compile(optimizer='adam', loss='mse')
print(model.summary())

# ----------------------------------------
# 9. Train the Model
# ----------------------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# ----------------------------------------
# 10. Evaluate on Test Data
# ----------------------------------------
test_loss = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (MSE): {test_loss:.4f}")

# ----------------------------------------
# 11. Predict and Inverse Scale
# ----------------------------------------
predictions_scaled = model.predict(X_test)

# Reshape y_test for inverse_transform
y_test_reshaped   = y_test.reshape(-1, 1)
predictions_reshaped = predictions_scaled.reshape(-1, 1)

predictions_orig = scaler.inverse_transform(predictions_reshaped)
y_test_orig      = scaler.inverse_transform(y_test_reshaped)

# ----------------------------------------
# 12. Compute Error Metrics in Original Scale
# ----------------------------------------
transformer_mae  = mean_absolute_error(y_test_orig, predictions_orig)
transformer_mse  = mean_squared_error(y_test_orig, predictions_orig)
transformer_rmse = np.sqrt(transformer_mse)
print(f"Transformer - Mean Absolute Error (MAE): {transformer_mae:.4f} MW")
print(f"Transformer - Root Mean Squared Error (RMSE): {transformer_rmse:.4f} MW")

# ----------------------------------------
# 13. Plot Predictions vs. Actual
# ----------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(y_test_orig, label='Actual', alpha=0.8)
plt.plot(predictions_orig, label='Predicted', alpha=0.8, linestyle='--')
plt.title("Transformer Predictions vs Actual Values (Original Scale)")
plt.xlabel("Time Step")
plt.ylabel("Electricity Consumption (MW)")
plt.legend()
plt.show()


## CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, losses, utils

model_CNN = models.Sequential([
    layers.Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(lookback, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)  # Final output for regression
])

model_CNN.summary()


In [ ]:
from keras.callbacks import EarlyStopping

model_CNN.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mae',           # or 'mse' - depends on preference
    metrics=['mae', 'mse']
)


early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model_CNN.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
test_loss, test_mae, test_mse = model_CNN.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE: {test_mae:.4f}, Test MSE: {test_mse:.4f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

test_predictions  = model_CNN.predict(X_test)

y_test_reshaped  = y_test.reshape(-1, 1)

test_preds_inv  = scaler.inverse_transform(test_predictions)

y_test_inv  = scaler.inverse_transform(y_test_reshaped)

mae_test  = mean_absolute_error(y_test_inv, test_preds_inv)
mse_test  = mean_squared_error(y_test_inv, test_preds_inv)
rmse_test = np.sqrt(mse_test)

print(f"Test   MAE: {mae_test:.4f},  RMSE: {rmse_test:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test_inv, label='Actual', alpha=0.8)
plt.plot(test_preds_inv, label='Predicted', alpha=0.8, linestyle='--')
plt.title("CNN: Actual vs. Predicted (Test)")
plt.xlabel("Time Steps (Test Set)")
plt.ylabel("Electricity Consumption (MW)")
plt.legend()
plt.grid()
plt.show()


## LSTM

In [ ]:
model_LSTM = models.Sequential()

model_LSTM.add(layers.LSTM(64, input_shape=(lookback, 1), return_sequences=True))
model_LSTM.add(layers.LSTM(32, return_sequences=True))
model_LSTM.add(layers.Dropout(0.2))
model_LSTM.add(layers.LSTM(32))
model_LSTM.add(layers.Dropout(0.2))

model_LSTM.add(layers.Dense(16, activation='relu'))
model_LSTM.add(layers.Dense(1))

model_LSTM.summary()

In [ ]:
model_LSTM.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mae',
    metrics=['mae', 'mse']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model_LSTM.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
test_loss, test_mae, test_mse = model_LSTM.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE: {test_mae:.4f} | Test MSE: {test_mse:.4f}")

In [ ]:
test_predictions = model_LSTM.predict(X_test)

y_test_reshaped   = y_test.reshape(-1, 1)         # shape (N,1)
test_preds_reshaped = test_predictions.reshape(-1, 1)

y_test_inv    = scaler.inverse_transform(y_test_reshaped)
test_preds_inv= scaler.inverse_transform(test_preds_reshaped)

lstm_mae_test  = mean_absolute_error(y_test_inv, test_preds_inv)
lstm_mse_test  = mean_squared_error(y_test_inv, test_preds_inv)
lstm_rmse_test = np.sqrt(mse_test)

print("\n--- LSTM Model Performance on Test Set (Original Scale) ---")
print(f"MAE: {lstm_mae_test:.4f}")
print(f"MSE: {lstm_mse_test:.4f}")
print(f"RMSE: {lstm_rmse_test:.4f}")


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test_inv, label='Actual', alpha=0.8)
plt.plot(test_preds_inv, label='Predicted', alpha=0.8, linestyle='--')
plt.title("LSTM Model: Actual vs. Predicted (Test Set)")
plt.xlabel("Time Steps (Test)")
plt.ylabel("Electricity Consumption (MW)")
plt.legend()
plt.grid()
plt.show()


## Final results and Visualisations

In [ ]:
# Print the extracted values
print(f"GRU - MAE: {gru_mae}, RMSE: {gru_rmse}")
print(f"Transformer - MAE: {transformer_mae}, RMSE: {transformer_rmse}")
print(f"CNN - MAE: {mae_test}, RMSE: {rmse_test}")
print(f"LSTM - MAE: {lstm_mae_test}, RMSE: {lstm_rmse_test}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define your metrics
models = ["GRU", "Transformer", "CNN", "LSTM"]
mae_values = [gru_mae, transformer_mae, mae_test, lstm_mae_test]
rmse_values = [gru_rmse, transformer_rmse, rmse_test, lstm_rmse_test]

# 2. Rank each metric (1 = best, higher = worse)
mae_ranks = np.argsort(mae_values) + 1
rmse_ranks = np.argsort(rmse_values) + 1

# 3. Create a grouped bar chart
x = np.arange(len(models))  # Label locations
width = 0.35  # Width of bars

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, mae_values, width, label='MAE', color='#1f77b4', edgecolor='black')
rects2 = ax.bar(x + width/2, rmse_values, width, label='RMSE', color='#ff7f0e', edgecolor='black')

# 4. Add axis labels, title, and grid lines
ax.set_ylabel('Metric Value', fontsize=12, fontweight='bold')
ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_title('Comparison of Model Performance Metrics', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# 5. Annotate Bars with their values + rank
def autolabel(rects, values, ranks, color):
    """
    Attach a text label above each bar, displaying the value
    and the rank in parentheses.
    """
    for idx, rect in enumerate(rects):
        height = rect.get_height()
        ax.annotate(f'{values[idx]:.2f}\n(rank {ranks[idx]})',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),  # Vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=10, color=color, fontweight='bold')

#autolabel(rects1, mae_values, mae_ranks, color='blue')
#autolabel(rects2, rmse_values, rmse_ranks, color='darkorange')

# 6. Adjust y-limit to fit labels
max_value = max(max(mae_values), max(rmse_values))
ax.set_ylim([0, max_value * 1.2])  # Extra space above bars

# 7. Improve aesthetics
plt.tight_layout()
plt.show()


## ARIMA

In [ ]:
!pip install pmdarima
import pmdarima as pm

In [ ]:
X_train_2d = X_train.reshape(X_train.shape[0], -1)


In [ ]:
model = pm.auto_arima(y_train, X=X_train_2d, seasonal=True, trace=True, error_action='ignore', suppress_warnings=True, stepwise=True)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=-524474.587, Time=131.27 sec


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-524482.587, Time=108.14 sec


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-524480.609, Time=115.22 sec


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-524484.551, Time=114.14 sec


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=-524478.566, Time=130.69 sec

Best model:  ARIMA(0,1,0)(0,0,0)[0]          
Total fit time: 716.222 seconds


In [ ]:
X_test_2d = X_test.reshape(X_test.shape[0], -1)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
X_val_2d = X_val.reshape(X_val.shape[0], -1)
y_val_2d = y_val.reshape(y_val.shape[0], -1)

X_train_linear_2d = np.concatenate([X_train_2d, X_val_2d], axis=0) # Concatenate along the first axis (rows)
y_train_linear_2d = np.concatenate([y_train, y_val], axis=0)

In [ ]:
arima_predictions = model.predict(n_periods=len(X_test), X=X_test_2d)

mae_scaled_arima = mean_absolute_error(y_test, arima_predictions)
mse_scaled_arima = mean_squared_error(y_test, arima_predictions)
rmse_scaled_arima = np.sqrt(mse_scaled_arima)

print(f"ARIMA - MAE (Scaled): {mae_scaled_arima:.4f}")
print(f"ARIMA - RMSE (Scaled): {rmse_scaled_arima:.4f}")
print(f"ARIMA - MAE (Unscaled): {mae_scaled_arima:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


ARIMA - MAE (Scaled): 0.0520
ARIMA - RMSE (Scaled): 0.0666
ARIMA - MAE (Unscaled): 0.0520


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
mae_unscaled_arima = mean_absolute_error(y_test_inv, arima_predictions)
mse_unscaled_arima = mean_squared_error(y_test_inv, arima_predictions)
rmse_unscaled_arima = np.sqrt(mse_unscaled_arima)

print(f"ARIMA - MAE (Unscaled): {mae_unscaled_arima:.4f}")
print(f"ARIMA - RMSE (Unscaled): {rmse_unscaled_arima:.4f}")
print(f"ARIMA - MSE (Unscaled): {mse_unscaled_arima:.4f}")

ARIMA - MAE (Unscaled): 2004.6156
ARIMA - RMSE (Unscaled): 2039.5220
ARIMA - MSE (Unscaled): 4159649.9528


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train_linear_2d, y_train_linear_2d)

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(random_state=42)

In [ ]:
rf_predictions = model_rf.predict(X_test_2d)

mae_rf = mean_absolute_error(y_test, rf_predictions)
mse_rf = mean_squared_error(y_test, rf_predictions)
rmse_rf = np.sqrt(mse_rf)

print(f"Random Forest - MAE: {mae_rf:.4f}")
print(f"Random Forest - RMSE: {rmse_rf:.4f}")
print(f"Random Forest - MSE: {mse_rf:.4f}")

Random Forest - MAE: 0.0074
Random Forest - RMSE: 0.0105
Random Forest - MSE: 0.0001


In [ ]:
mae_rf_unscaled = mean_absolute_error(y_test_inv, rf_predictions)
mse_rf_unscaled = mean_squared_error(y_test_inv, rf_predictions)
rmse_rf_unscaled = np.sqrt(mse_rf_unscaled)

print(f"Random Forest - MAE (Unscaled): {mae_rf_unscaled:.4f}")
print(f"Random Forest - RMSE (Unscaled): {rmse_rf_unscaled:.4f}")
print(f"Random Forest - MSE (Unscaled): {mse_rf_unscaled:.4f}")

Random Forest - MAE (Unscaled): 2004.5984
Random Forest - RMSE (Unscaled): 2039.4960
Random Forest - MSE (Unscaled): 4159544.0602
